# College Admission OpenEnv — TRL Training (Kaggle / Local GPU)

This notebook runs `train_trl_kaggle.py` end-to-end with:
- Unsloth-first QLoRA loading
- Weights & Biases tracking
- Qwen2.5 1.5B Instruct (closest 1B-class checkpoint)
- Random vs untrained vs trained policy evaluation


In [ ]:
import platform
import subprocess
import sys

if sys.version_info >= (3, 13):
    raise RuntimeError(
        "This training stack requires Python <= 3.12 (recommended 3.10/3.11). "
        "Switch the notebook kernel to your college-admission environment first."
    )

packages = [
    "torch>=2.4.0",
    "transformers>=4.46.0",
    "trl>=0.11.0",
    "peft>=0.13.0",
    "accelerate>=0.34.0",
    "bitsandbytes>=0.43.0",
    "datasets>=2.20.0",
    "huggingface_hub>=0.25.0",
    "pandas>=2.1.0",
    "matplotlib>=3.8.0",
    "requests>=2.31.0",
    "wandb>=0.17.0",
    "ipykernel>=6.29.0",
]

if platform.system().lower() != "windows":
    packages.insert(0, "unsloth>=2025.1.0")
else:
    packages = [pkg for pkg in packages if not pkg.startswith("bitsandbytes")]
    print("Windows kernel detected: skipping unsloth/bitsandbytes install. Script will auto-fallback to Transformers backend.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "wheel", "setuptools"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependency installation complete.")


In [ ]:
import os
import platform
import subprocess
import sys

# Required runtime config
os.environ["COLLEGE_ENV_BASE_URL"] = "https://Knight09-college-admission-env.hf.space"
if platform.system().lower() == "windows":
    os.environ["BASE_MODEL_NAME"] = "Qwen/Qwen2.5-1.5B-Instruct"
    os.environ["OUTPUT_DIR"] = "./trl_runs/college_env_qwen25_15b"
else:
    os.environ["BASE_MODEL_NAME"] = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
    os.environ["OUTPUT_DIR"] = "./trl_runs/college_env_qwen25_15b_unsloth"

# Training/eval knobs
os.environ["MAX_SEQ_LENGTH"] = "512"
os.environ["TRAIN_EPISODES_PER_TEMPLATE"] = "110"
os.environ["MAX_TRAIN_STEPS"] = "600"
os.environ["TRAIN_BATCH_SIZE"] = "2"
os.environ["GRAD_ACCUM_STEPS"] = "8"
os.environ["EVAL_EPISODES_PER_TASK"] = "12"

# W&B tracking
os.environ["USE_WANDB"] = "1"
os.environ.setdefault("WANDB_PROJECT", "college-admission-openenv")
os.environ.setdefault("WANDB_RUN_NAME", "qwen25_15b_unsloth_trl")
# Optional if not already logged in:
# os.environ["WANDB_API_KEY"] = "<your_wandb_api_key>"
# os.environ["WANDB_ENTITY"] = "<your_team_or_username>"

# Optional HF Hub push
# os.environ["PUSH_TO_HUB"] = "1"
# os.environ["HF_TOKEN"] = "<your_hf_token>"
# os.environ["HUB_MODEL_REPO"] = "<username>/<model-repo-name>"

# Optional HF Space dashboard push
# os.environ["PUSH_TO_SPACE_DASHBOARD"] = "1"
# os.environ["HF_SPACE_REPO"] = "<username>/<space-repo-name>"

cmd = [sys.executable, "train_trl_kaggle.py"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import Image, display

output_dir = Path(os.environ.get("OUTPUT_DIR", "./trl_runs/college_env_qwen25_15b_unsloth"))
artifact_dir = output_dir / "artifacts"

summary_path = artifact_dir / "evaluation_summary.csv"
episodes_path = artifact_dir / "evaluation_episodes.csv"
print("Summary:", summary_path)
print("Episodes:", episodes_path)
print("\nEvaluation summary table:")
display(pd.read_csv(summary_path).head(30))

for img_name in ["training_loss_curve.png", "episode_return_curve.png", "task_score_bar.png"]:
    img_path = artifact_dir / img_name
    print(f"\n{img_name} -> {img_path}")
    display(Image(filename=str(img_path)))
